# Laboratori 09: Numrat pseudorastësorë dhe kampionimi

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Si shndërrohet një varg determinist uniform në mostra me shpërndarjen e kërkuar?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- shpjegojë pse gjeneratori pseudorastësor është determinist;
- llogarisë me dorë hapat e një gjeneratori kongruencial linear;
- kontrollojë uniformitetin dhe varësinë serike;
- nxjerrë metodën e transformimit të anasjelltë;
- zbatojë Box–Muller dhe kampionimin me refuzim;
- nxjerrë koston e pritshme nga probabiliteti i pranimit;
- trajtojë në mënyrë të sigurt vlerat kufitare të uniformes.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Gjeneratori uniform është burimi bazë; metodat e kampionimit e transformojnë atë në shpërndarje të tjera. Duhet të ndahen cilësia e burimit pseudorastësor nga korrektësia e transformimit. Një formulë e saktë e transformimit nuk shpëton një gjenerator me periodë të shkurtër ose korrelacione të forta.

Gjatë laboratorit ruani farën dhe emrin e gjeneratorit. Mos e rivendosni farën para çdo thirrjeje, sepse do të prodhonit blloqe identike. Një farë e vetme në fillim dhe kalimi i të njëjtit objekt `rng` te funksionet e bën rrjedhën të qartë.


## 1. Pseudorastësia dhe gjendja

Një gjenerator pseudorastësor është një algoritëm determinist që përditëson një gjendje. E njëjta farë prodhon të njëjtin varg; kjo është thelbësore për riprodhueshmërinë. Cilësia nuk gjykohet nga pamja “e çrregullt”, por nga teste statistike, perioda dhe sjellja në dimensione të larta.

Nuk duhet përdorur një gjenerator i thjeshtë pedagogjik për kriptografi ose simulime kërkimore të ndjeshme. Ai shërben për të kuptuar mekanizmin.


## 2. Gjeneratori kongruencial linear me dorë

Përcaktojmë

$$X_{n+1}=(aX_n+c)\bmod m,\qquad U_n=\frac{X_n}{m}.$$

Me $m=9$, $a=4$, $c=1$, $X_0=0$:

$$X_1=1,$$
$$X_2=(4\cdot1+1)\bmod9=5,$$
$$X_3=(4\cdot5+1)\bmod9=3,$$
$$X_4=(4\cdot3+1)\bmod9=4.$$

Vazhdoni derisa një gjendje të përsëritet. Përsëritja e gjendjes nënkupton përsëritjen e gjithë vargut të ardhshëm.


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


In [ ]:
def lcg(seed,a=4,c=1,m=9,n=20):
    x=int(seed); out=[]
    for _ in range(n):
        x=(a*x+c)%m
        out.append(x)
    return np.array(out)

v=lcg(0)
print('gjendjet:',v)
print('uniformet:',v/9)


In [ ]:
def perioda_lcg(seed,a,c,m):
    pare={seed:0}; x=seed
    for k in range(1,m+2):
        x=(a*x+c)%m
        if x in pare:
            return k-pare[x],pare[x],x
        pare[x]=k
    return None

for params in [(4,1,9),(5,1,16),(5,0,16)]:
    print(params,perioda_lcg(0,*params))


## 3. Kontrolli i një vargu uniform

Për $U\sim U(0,1)$,

$$\mathbb E[U]=\frac12,\qquad \operatorname{Var}(U)=\frac1{12}.$$

Një histogram kontrollon vetëm shpërndarjen njëdimensionale. Grafiku $(U_n,U_{n+1})$ dhe autokorrelacioni kontrollojnë varësinë serike. Kalimi i një testi nuk provon rastësi të përsosur; dështimi është sinjal i qartë problemi.


In [ ]:
u=rng.random(100_000)
print('mesatarja=',u.mean(),'pritja=.5')
print('varianca=',u.var(ddof=1),'pritja=',1/12)
for lag in [1,2,5,10]:
    rho=np.corrcoef(u[:-lag],u[lag:])[0,1]
    print('lag',lag,'rho=',rho)


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].hist(u,bins=50,density=True); ax[0].axhline(1,color='k',ls='--')
ax[0].set(xlabel='u',ylabel='dendësia',title='Histogrami')
ax[1].scatter(u[:3000],u[1:3001],s=3,alpha=.4)
ax[1].set(xlabel='$U_n$',ylabel='$U_{n+1}$',title='Çiftet e njëpasnjëshme')
for a in ax:a.grid(alpha=.3)
plt.show()


## 4. Transformimi i anasjelltë

Nëse $U\sim U(0,1)$ dhe $F$ është funksioni kumulativ i synuar, atëherë

$$X=F^{-1}(U)$$

ka shpërndarjen $F$. Për shpërndarjen eksponenciale me parametër $\lambda$,

$$F(x)=1-e^{-\lambda x},\qquad x\geq0.$$

Duke vendosur $U=F(X)$:

$$X=-\frac1\lambda\ln(1-U).$$

Përdorimi i `log1p(-u)` është më i saktë kur $u$ është i vogël. Uniformja duhet të jetë në intervalin e hapur për të shmangur $\ln0$.


In [ ]:
def eksponenciale_anasjellte(n,lam=2.0,rng=rng):
    u=rng.random(n)
    return -np.log1p(-u)/lam

x=eksponenciale_anasjellte(200_000,lam=2)
print('mesatarja=',x.mean(),'teorike=',.5,'varianca=',x.var(),'teorike=',.25)


## 5. Box–Muller për normalen

Dy uniforme të pavarura japin

$$Z_1=\sqrt{-2\ln U_1}\cos(2\pi U_2),$$

$$Z_2=\sqrt{-2\ln U_1}\sin(2\pi U_2).$$

Transformimi polar përdor simetrinë rrethore të normales dypërmasore. Kostoja për dy mostra është konstante, por funksionet transcendente janë më të shtrenjta se mbledhjet.


In [ ]:
def box_muller(n,rng=rng):
    m=(n+1)//2
    u1=np.maximum(rng.random(m),np.finfo(float).tiny)  # shmang log(0)
    u2=rng.random(m)
    r=np.sqrt(-2*np.log(u1)); theta=2*np.pi*u2
    z=np.concatenate([r*np.cos(theta),r*np.sin(theta)])
    return z[:n]

z=box_muller(200_000)
print('mesatarja=',z.mean(),'devijimi=',z.std(ddof=1))


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 6.1 Kostoja e kampionimit

Transformimi i anasjelltë dhe Box–Muller prodhojnë një numër konstant mostrash për hap, prandaj për $N$ mostra kostoja është $\Theta(N)$.

Në kampionimin me refuzim, nëse probabiliteti i pranimit është $p_a$, numri i propozimeve për një pranim ka pritje $1/p_a$. Për $N$ mostra:

$$\mathbb E[K]=\frac{N}{p_a},$$

prandaj kostoja e pritshme është $\Theta(N/p_a)$. Një mbështjellëse e dobët ul $p_a$ dhe rrit drejtpërdrejt kohën.


## 7. Kampionimi me refuzim: shpërndarje trekëndore

Synojmë dendësinë $f(x)=2x$ në $[0,1]$. Propozojmë $X\sim U(0,1)$ dhe $Y\sim U(0,2)$. Pranojmë kur $Y\leq2X$. Zona nën kurbë ka sipërfaqe $1$, ndërsa drejtkëndëshi ka sipërfaqe $2$, prandaj

$$p_a=\frac12.$$

Presim rreth $2N$ propozime për $N$ pranime.


In [ ]:
def refuzim_trekendor(N,bllok=4096,rng=rng):
    pranuar=[]; propozime=0
    while sum(len(a) for a in pranuar)<N:
        x=rng.random(bllok); y=2*rng.random(bllok)
        pranuar.append(x[y<=2*x]); propozime+=bllok
    mostra=np.concatenate(pranuar)[:N]
    return mostra,N/propozime

x,pa=refuzim_trekendor(100_000)
print('pranimi i matur=',pa,'teorik=.5','mesatarja=',x.mean(),'teorike=2/3')


In [ ]:
Ns=np.array([10_000,20_000,40_000,80_000,160_000]); ts=[]
for N in Ns:
    t0=perf_counter(); refuzim_trekendor(int(N)); ts.append(perf_counter()-t0)
print('pjerrësia e kohës=',np.polyfit(np.log(Ns),np.log(ts),1)[0])


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 8.1 Qëndrueshmëria dhe riprodhueshmëria

Rreziqet kryesore janë:

- përdorimi i $U=0$ në $\ln U$;
- humbja e saktësisë në $\ln(1-U)$, e shmangur me `log1p(-U)`;
- ripërdorimi aksidental i së njëjtës farë në rrjedha që supozohen të pavarura;
- ndryshimi i algoritmit të gjeneratorit midis versioneve pa e dokumentuar.

Fara identifikon realizimin; ajo nuk është një test cilësie. Për punë paralele përdoren rrjedha të ndara të projektuara nga biblioteka, jo fara të njëpasnjëshme të zgjedhura pa analizë.


In [ ]:
u_vogel=np.array([1e-16,1e-12,1e-8])
naive=np.log(1-u_vogel)
stabile=np.log1p(-u_vogel)
for u,a,b in zip(u_vogel,naive,stabile):
    print(f'u={u:.0e}, log(1-u)={a:.18e}, log1p(-u)={b:.18e}')


## 9. Shembulli i plotë dhe punishte dinamike

Gjeneroni mostra eksponenciale, normale dhe trekëndore. Për secilën krahasoni mesataren, variancën dhe histogramin me teorinë. Në punishte ndryshoni $N$ dhe shihni si stabilizohet histogrami; mos ndryshoni shpërndarjen dhe numrin e mostrave njëkohësisht.


In [ ]:
def histogram_dinamik(gjenerator,N=20_000,bllok=500,DINAMIK=False):
    mostra=[]; fig,ax=plt.subplots(figsize=(7,4))
    for _ in range(N//bllok):
        mostra.extend(gjenerator(bllok))
        if DINAMIK:
            ax.clear(); ax.hist(mostra,bins=40,density=True,alpha=.75)
            ax.set(title=f'N={len(mostra)}'); plt.pause(.03)
    if not DINAMIK:
        ax.hist(mostra,bins=40,density=True,alpha=.75); ax.set(title=f'N={len(mostra)}')
    ax.grid(alpha=.3); plt.show()
    return np.array(mostra)

mostra=histogram_dinamik(lambda n:eksponenciale_anasjellte(n,2),DINAMIK=False)


### Nga CDF-ja te dendësia e mostrës së transformuar

Nëse $X=F^{-1}(U)$, atëherë

$$\Pr(X\leq x)=\Pr(U\leq F(x))=F(x),$$

sepse $U$ është uniforme në $[0,1]$. Ky argument provon metodën e anasjelltë pa u mbështetur te histogrami. Histogrami është vetëm një kontroll empirik me numër të fundëm mostrash.


In [ ]:
# Krahasim vizual midis LCG-së së vogël dhe PCG64 të NumPy-t.
u_lcg=lcg(0,a=5,c=1,m=256,n=256)/256
u_pcg=rng.random(256)
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].scatter(u_lcg[:-1],u_lcg[1:],s=10); ax[0].set_title('LCG i vogël')
ax[1].scatter(u_pcg[:-1],u_pcg[1:],s=10); ax[1].set_title('PCG64')
for a in ax:a.set(xlabel='$U_n$',ylabel='$U_{n+1}$');a.grid(alpha=.3)
plt.show()


### Testi $\chi^2$ si kontroll i thjeshtë uniformiteti

Ndajmë $[0,1)$ në $k$ kuti. Nën uniformitetin priten $E=N/k$ vlera në secilën. Statistika

$$\chi^2=\sum_{j=1}^{k}\frac{(O_j-E)^2}{E}$$

duhet të jetë e krahasueshme me $k-1$, por një vlerë e vetme nuk provon cilësinë e gjeneratorit. Testi varet nga $N$, $k$ dhe niveli i zgjedhur i rëndësisë.


In [ ]:
for name,u_test in [('PCG',rng.random(100000)),('LCG',lcg(0,a=1103515245,c=12345,m=2**31,n=100000)/(2**31))]:
    counts,_=np.histogram(u_test,bins=20,range=(0,1)); E=len(u_test)/20
    chi=np.sum((counts-E)**2/E)
    print(name,'chi2=',chi,'shkallë lirie=19')


### Efikasiteti nuk është vetëm koha e murit

Për refuzimin, norma e pranimit përcakton sa vlerësime hidhen poshtë. Për transformimin e anasjelltë, kostoja varet nga disponueshmëria e $F^{-1}$. Për Box–Muller-in, dy uniforme japin dy normale, por kërkohen logaritëm, rrënjë dhe funksione trigonometrike. Zgjedhja duhet të balancojë saktësinë, koston dhe kompleksitetin e zbatimit.


In [ ]:
# Variabiliteti i normës së pranimit në eksperimente të përsëritura.
rates=[]
for _ in range(40):
    _,rate=refuzim_trekendor(5000,bllok=512); rates.append(rate)
print('pranimi mesatar=',np.mean(rates),'devijimi=',np.std(rates,ddof=1))


### Pyetje për vetëkontroll

1. Pse riprodhueshmëria kërkon më shumë se vetëm farën?
2. Çfarë kontrollon grafiku $(U_n,U_{n+1})$ që nuk e kontrollon histogrami?
3. Pse `log1p(-u)` preferohet ndaj `log(1-u)`?
4. Si lidhet norma e pranimit me kompleksitetin e pritshëm?


# Pjesa II — GNU Octave

## 10. LCG dhe transformimi i anasjelltë

```octave
a=4; c=1; m=9; x=0; N=20; X=zeros(1,N);
for n=1:N
    x=mod(a*x+c,m); X(n)=x;
end
disp(X); disp(X/m);

rand('seed',2026); lambda=2; u=rand(100000,1);
xe=-log1p(-u)/lambda;
fprintf('mesatarja %.6f, varianca %.6f\n',mean(xe),var(xe));
```

## 11. Box–Muller dhe refuzimi

```octave
N=100000; u1=max(rand(ceil(N/2),1),realmin); u2=rand(ceil(N/2),1);
r=sqrt(-2*log(u1)); z=[r.*cos(2*pi*u2);r.*sin(2*pi*u2)]; z=z(1:N);
fprintf('normalja: mu %.5f, sigma %.5f\n',mean(z),std(z));

pranuar=[]; propozime=0;
while length(pranuar)<N
    x=rand(4096,1); y=2*rand(4096,1);
    pranuar=[pranuar;x(y<=2*x)]; propozime=propozime+4096;
end
pranuar=pranuar(1:N);
fprintf('pranimi %.5f\n',N/propozime);
```


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = 'import numpy as np\nrng=np.random.default_rng(2026)\nfor N in [1000,10000,100000]:\n u=rng.random(N); x=-np.log1p(-u)/2\n print(N,x.mean(),x.var())\n'
script_path = script_dir / 'laboratori_09_pseudorastesore_kampionimi.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_09_pseudorastesore_kampionimi.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Llogarita me dorë hapat e LCG-së.
- [ ] Kontrollova më shumë se histogramin.
- [ ] Nxora koston $N/p_a$ të refuzimit.
- [ ] Trajtova në mënyrë të sigurt logaritmin pranë kufirit.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
